# Catalog Data Interpretation

Pulls out schema fields from `data/catalog.jsonl`, checks completeness per field, and checks how well represented each category is.


In [1]:
import json
from collections import Counter
from pathlib import Path

CATALOG_PATH = Path("../data/catalog.jsonl")

products = []
with CATALOG_PATH.open(encoding="utf-8") as f:
    for line in f:
        products.append(json.loads(line))

total = len(products)
print(f"Loaded {total} products")

Loaded 50000 products


## Pull out schema fields

`parent_asin`, `title`, `store`, `categories`, `price`, `details`.


In [2]:
def extract_fields(product: dict) -> dict:
    return {
        "parent_asin": product.get("parent_asin"),
        "title": product.get("title"),
        "store": product.get("store"),
        "categories": product.get("categories"),
        "price": product.get("price"),
        "details": product.get("details"),
    }

extracted = [extract_fields(p) for p in products]
extracted[0]

{'parent_asin': 'B07K34RX5J',
 'title': 'Kandinsky Statement Earrings for Women by Spirit Hoops, Fabric, Lightweight Drop and Dangle Stainless Steel Hoop Earrings for Women Fashion, Artsy',
 'store': 'Spirit Hoops',
 'categories': ['Clothing, Shoes & Jewelry',
  'Women',
  'Jewelry',
  'Earrings',
  'Hoop'],
 'price': None,
 'details': {'Is Discontinued By Manufacturer': 'No',
  'Product Dimensions': '1.97 x 1.97 x 0.08 inches; 0.5 Ounces',
  'Department': 'Womens',
  'Date First Available': 'October 29, 2018',
  'Manufacturer': 'Spirit Hoops'}}

## Field completeness: `parent_asin`


In [3]:
present = sum(1 for p in extracted if p["parent_asin"])
print(f"parent_asin present: {present}/{total} ({present/total:.1%})")

parent_asin present: 50000/50000 (100.0%)


## Field completeness: `title`


In [4]:
present = sum(1 for p in extracted if p["title"])
print(f"title present: {present}/{total} ({present/total:.1%})")

title present: 49998/50000 (100.0%)


## Field completeness: `store`


In [5]:
present = sum(1 for p in extracted if p["store"])
print(f"store present: {present}/{total} ({present/total:.1%})")

store present: 49686/50000 (99.4%)


## Field completeness: `categories`


In [6]:
present = sum(1 for p in extracted if p["categories"])
print(f"categories present: {present}/{total} ({present/total:.1%})")

categories present: 50000/50000 (100.0%)


## Field completeness: `price`


In [7]:
present = sum(1 for p in extracted if p["price"] is not None)
print(f"price present: {present}/{total} ({present/total:.1%})")
print(f"price missing: {total - present}/{total} ({(total - present)/total:.1%})")

price present: 10527/50000 (21.1%)
price missing: 39473/50000 (78.9%)


## Field completeness: `details`


In [9]:
present = sum(1 for p in extracted if p["details"])
print(f"details non-empty: {present}/{total} ({present/total:.1%})")

key_counts = Counter()
for p in extracted:
    if p["details"]:
        key_counts.update(p["details"].keys())

print("\nMost common detail keys:")
for key, count in key_counts.most_common(10):
    print(f"  {key}: {count}/{total} ({count/total:.1%})")

details non-empty: 48330/50000 (96.7%)

Most common detail keys:
  Date First Available: 46886/50000 (93.8%)
  Department: 43582/50000 (87.2%)
  Item model number: 27729/50000 (55.5%)
  Package Dimensions: 27061/50000 (54.1%)
  Manufacturer: 23512/50000 (47.0%)
  Is Discontinued By Manufacturer: 13070/50000 (26.1%)
  Product Dimensions: 10210/50000 (20.4%)
  Item Weight: 3243/50000 (6.5%)
  Color: 2439/50000 (4.9%)
  Brand: 2328/50000 (4.7%)


## Category representation

`categories` is a list like `["Clothing, Shoes & Jewelry", "Women", "Jewelry", "Earrings", "Hoop"]`. Index 1 is the top-level department (Women/Men/Boys/Girls/Unisex etc), index -1 is the leaf/finest-grained category.


In [10]:
department_counts = Counter()
for p in extracted:
    cats = p["categories"] or []
    department = cats[1] if len(cats) > 1 else "(unknown)"
    department_counts[department] += 1

print("Department representation (categories[1]):")
for department, count in department_counts.most_common():
    print(f"  {department}: {count}/{total} ({count/total:.1%})")

Department representation (categories[1]):
  Women: 26406/50000 (52.8%)
  Men: 9901/50000 (19.8%)
  Novelty & More: 3376/50000 (6.8%)
  Girls: 1716/50000 (3.4%)
  Westlake: 1136/50000 (2.3%)
  Boot Shop: 1131/50000 (2.3%)
  Sport Specific Clothing: 1114/50000 (2.2%)
  Boys: 1101/50000 (2.2%)
  Baby: 1031/50000 (2.1%)
  Luggage & Travel Gear: 976/50000 (2.0%)
  Costumes & Accessories: 937/50000 (1.9%)
  Shoe, Jewelry & Watch Accessories: 436/50000 (0.9%)
  Toddler Test: 48/50000 (0.1%)
  Kids Shoes Union: 36/50000 (0.1%)
  Top 50 by Product Type: 32/50000 (0.1%)
  Uniforms, Work & Safety: 31/50000 (0.1%)
  Swimwear TEST: 29/50000 (0.1%)
  Plus-Size Fashion: 29/50000 (0.1%)
  Women's Plus-Size Apparel: 27/50000 (0.1%)
  Customers' Most-Loved: Sweaters Under $30 pASIN Test: 24/50000 (0.0%)
  Everyday Apparel Essentials: 20/50000 (0.0%)
  Graduation Cohort 4 - Daily: 19/50000 (0.0%)
  MFN ONLY V2: 18/50000 (0.0%)
  PattyBoutik Apparel: 16/50000 (0.0%)
  Women's Halloween Costumes: 15/50000

In [11]:
leaf_counts = Counter()
for p in extracted:
    cats = p["categories"] or []
    leaf = cats[-1] if cats else "(unknown)"
    leaf_counts[leaf] += 1

print(f"Distinct leaf categories: {len(leaf_counts)}")
print("\nTop 20 leaf categories by representation:")
for leaf, count in leaf_counts.most_common(20):
    print(f"  {leaf}: {count}/{total} ({count/total:.1%})")

Distinct leaf categories: 800

Top 20 leaf categories by representation:
  T-Shirts: 2807/50000 (5.6%)
  Shoes: 1299/50000 (2.6%)
  Westlake: 1136/50000 (2.3%)
  Casual: 1099/50000 (2.2%)
  Wrist Watches: 1034/50000 (2.1%)
  Fashion Sneakers: 1017/50000 (2.0%)
  Flats: 927/50000 (1.9%)
  Blouses & Button-Down Shirts: 691/50000 (1.4%)
  Loafers & Slip-Ons: 665/50000 (1.3%)
  Dresses: 656/50000 (1.3%)
  Pumps: 630/50000 (1.3%)
  Sets: 610/50000 (1.2%)
  Sandals: 586/50000 (1.2%)
  Platforms & Wedges: 545/50000 (1.1%)
  Sunglasses: 540/50000 (1.1%)
  Slippers: 538/50000 (1.1%)
  Pendant Necklaces: 531/50000 (1.1%)
  Road Running: 522/50000 (1.0%)
  Tunics: 521/50000 (1.0%)
  Drop & Dangle: 503/50000 (1.0%)


In [12]:
thin_leaves = [leaf for leaf, count in leaf_counts.items() if count < 30]
print(f"Leaf categories with fewer than 30 items (thin, poorly represented): {len(thin_leaves)}")
print(f"Items affected: {sum(leaf_counts[l] for l in thin_leaves)}/{total}")

Leaf categories with fewer than 30 items (thin, poorly represented): 541
Items affected: 4092/50000


## Summary

Re-run this notebook after any catalog update to re-check field completeness and category balance before relying on it for B2/B3 indexing decisions.


## All-in-one (copy/paste, single run for full output dump)


In [1]:
import json
from collections import Counter
from pathlib import Path

# --- Load catalog ---
CATALOG_PATH = Path("../data/catalog.jsonl")
products = []
with CATALOG_PATH.open(encoding="utf-8") as f:
    for line in f:
        products.append(json.loads(line))
total = len(products)
print(f"Loaded {total} products")

# --- Pull out schema fields (parent_asin, title, store, categories, price, details) ---
def extract_fields(product: dict) -> dict:
    return {
        "parent_asin": product.get("parent_asin"),
        "title": product.get("title"),
        "store": product.get("store"),
        "categories": product.get("categories"),
        "price": product.get("price"),
        "details": product.get("details"),
    }

extracted = [extract_fields(p) for p in products]
print("\nSample extracted record:")
print(extracted[0])

# --- Field completeness: parent_asin ---
present = sum(1 for p in extracted if p["parent_asin"])
print(f"\nparent_asin present: {present}/{total} ({present/total:.1%})")

# --- Field completeness: title ---
present = sum(1 for p in extracted if p["title"])
print(f"title present: {present}/{total} ({present/total:.1%})")

# --- Field completeness: store ---
present = sum(1 for p in extracted if p["store"])
print(f"store present: {present}/{total} ({present/total:.1%})")

# --- Field completeness: categories ---
present = sum(1 for p in extracted if p["categories"])
print(f"categories present: {present}/{total} ({present/total:.1%})")

# --- Field completeness: price ---
present = sum(1 for p in extracted if p["price"] is not None)
print(f"price present: {present}/{total} ({present/total:.1%})")
print(f"price missing: {total - present}/{total} ({(total - present)/total:.1%})")

# --- Field completeness: details (+ most common detail keys) ---
present = sum(1 for p in extracted if p["details"])
print(f"details non-empty: {present}/{total} ({present/total:.1%})")
key_counts = Counter()
for p in extracted:
    if p["details"]:
        key_counts.update(p["details"].keys())
print("Most common detail keys:")
for key, count in key_counts.most_common(10):
    print(f"  {key}: {count}/{total} ({count/total:.1%})")

# --- Category representation: department level (categories[1]) ---
department_counts = Counter()
for p in extracted:
    cats = p["categories"] or []
    department = cats[1] if len(cats) > 1 else "(unknown)"
    department_counts[department] += 1
print("\nDepartment representation (categories[1]):")
for department, count in department_counts.most_common():
    print(f"  {department}: {count}/{total} ({count/total:.1%})")

# --- Category representation: leaf level (categories[-1]) ---
leaf_counts = Counter()
for p in extracted:
    cats = p["categories"] or []
    leaf = cats[-1] if cats else "(unknown)"
    leaf_counts[leaf] += 1
print(f"\nDistinct leaf categories: {len(leaf_counts)}")
print("Top 20 leaf categories by representation:")
for leaf, count in leaf_counts.most_common(20):
    print(f"  {leaf}: {count}/{total} ({count/total:.1%})")

# --- Thin/poorly represented leaf categories (fewer than 30 items) ---
thin_leaves = [leaf for leaf, count in leaf_counts.items() if count < 30]
print(f"\nLeaf categories with fewer than 30 items (thin, poorly represented): {len(thin_leaves)}")
print(f"Items affected: {sum(leaf_counts[l] for l in thin_leaves)}/{total}")


Loaded 50000 products

Sample extracted record:
{'parent_asin': 'B07K34RX5J', 'title': 'Kandinsky Statement Earrings for Women by Spirit Hoops, Fabric, Lightweight Drop and Dangle Stainless Steel Hoop Earrings for Women Fashion, Artsy', 'store': 'Spirit Hoops', 'categories': ['Clothing, Shoes & Jewelry', 'Women', 'Jewelry', 'Earrings', 'Hoop'], 'price': None, 'details': {'Is Discontinued By Manufacturer': 'No', 'Product Dimensions': '1.97 x 1.97 x 0.08 inches; 0.5 Ounces', 'Department': 'Womens', 'Date First Available': 'October 29, 2018', 'Manufacturer': 'Spirit Hoops'}}

parent_asin present: 50000/50000 (100.0%)
title present: 49998/50000 (100.0%)
store present: 49686/50000 (99.4%)
categories present: 50000/50000 (100.0%)
price present: 10527/50000 (21.1%)
price missing: 39473/50000 (78.9%)
details non-empty: 48330/50000 (96.7%)
Most common detail keys:
  Date First Available: 46886/50000 (93.8%)
  Department: 43582/50000 (87.2%)
  Item model number: 27729/50000 (55.5%)
  Package Dim